# Imports


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"

ORDERS_PATH = RAW_DIR / "olist_orders_dataset.csv"
ITEMS_PATH = RAW_DIR / "olist_order_items_dataset.csv"
CUSTOMERS_PATH = RAW_DIR / "olist_customers_dataset.csv"
REVIEWS_PATH = RAW_DIR / "olist_order_reviews_dataset.csv"
PRODUCTS_PATH = RAW_DIR / "olist_products_dataset.csv"


# Load data


In [2]:
reviews_df = pd.read_csv(REVIEWS_PATH)
order_items_df = pd.read_csv(ITEMS_PATH)
orders_df = pd.read_csv(ORDERS_PATH)

orders_df["order_id"] = orders_df["order_id"].astype(str).str.strip()
reviews_df["order_id"] = reviews_df["order_id"].astype(str).str.strip()
order_items_df["order_id"] = order_items_df["order_id"].astype(str).str.strip()
orders_df["order_status"] = orders_df["order_status"].astype(str).str.strip()

# Delivered filtering

In [3]:
order_sizes = order_items_df.groupby("order_id").size()
single_orders = set(order_sizes[order_sizes == 1].index.astype(str))

reviews_1 = reviews_df[reviews_df["order_id"].astype(str).isin(single_orders)].copy()
oi_1 = order_items_df[order_items_df["order_id"].astype(str).isin(single_orders)].copy()

print(len(reviews_1))
print(len(reviews_df))
print(len(orders_df))


88699
99224
99441


# Map reviews -> items

In [4]:
df = (
    reviews_1[["order_id", "review_comment_message"]]
    .dropna()
    .merge(oi_1[["order_id", "product_id"]], on="order_id", how="inner")
    .rename(columns={"product_id": "item_id"})
)


print(df.isna().sum())
print(df.shape)
df.head()


order_id                  0
review_comment_message    0
item_id                   0
dtype: int64
(35489, 3)


,order_id,review_comment_message,item_id
0,658677c97b385a9be170737859d3511b,Recebi bem antes do prazo estipulado.,52c80cedd4e90108bf4fa6a206ef6b03
1,8e6bfb81e283fa7e4f11123a3fb894f1,Parabéns lojas lannister adorei comprar pela I...,3880d25d502b15b1de6fddc42ad1d67a
2,b9bf720beb4ab3728760088589c62129,aparelho eficiente. no site a marca do aparelh...,61a4100ccd6d9c4c808a1fd954ddb8ad
3,9d6f15f95d01e79bd1349cc208361f09,"Mas um pouco ,travando...pelo valor ta Boa.\r\n",acffe5d7cd56e6b564cf6841486644ff
4,e51478e7e277a83743b6f9991dbfa3fb,"Vendedor confiável, produto ok e entrega antes...",6871a3c157d6f51697e887f3c3598479


In [5]:
item_text = (
    df.groupby("item_id")["review_comment_message"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

item_text["review_comment_message"] = item_text["review_comment_message"].str.lower()
print(item_text.shape)

item_text["n_tokens"] = item_text["review_comment_message"].str.split().str.len()
item_text = item_text[item_text["n_tokens"] >= 8].copy()


def unique_ratio(text):
    toks = text.split()
    return len(set(toks)) / max(len(toks), 1)


item_text["uniq_ratio"] = item_text["review_comment_message"].apply(unique_ratio)
item_text = item_text[item_text["uniq_ratio"] >= 0.3].copy()
item_text = item_text.reset_index(drop=True).copy()
item_text.head()


(16308, 2)


,item_id,review_comment_message,n_tokens,uniq_ratio
0,0009406fd7479715e4bef61dd91f2462,meu produto não foi entregue até o momento!,8,1.000000
1,000b8f95fcb9e0096488278317764d19,"produto bom, mas o pegador da tampa é de plást...",39,0.871795
2,000d9be29b5207b54e86aa1b1ac54872,recebi antes do prazo e relógio é lindo,8,1.000000
3,001795ec6f1b187d37335e1c4704762e,cancelei a compra e mesmo assim me entregaram ...,34,0.852941
4,001b237c0e9bb435f2e54071129237e9,só faltou o outro que foi comprado junto,8,1.000000


# Build TF-IDF matrix

## Stopwords

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")
# portuguese_stopword = stopwords.words("portuguese")

portuguese_stopword = set(stopwords.words("portuguese"))
portuguese_stopword.discard("não")
portuguese_stopword.discard("nao")
portuguese_stopword.discard("nunca")
portuguese_stopword.discard("jamais")
portuguese_stopword.discard("nem")

portuguese_stopword = list(portuguese_stopword)

extra_generic = {
    "produto",
    "produtos",
    "loja",
    "entrega",
    "rápida",
    "rapida",
    "chegou",
    "excelente",
    "ótima",
    "otima",
    "bom",
    "boa",
    "qualidade",
    "recomendo",
    "perfeito",
    "ok",
    "muito",
    "bem",
    "super",
}

stop_pt = set(portuguese_stopword) | extra_generic


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Jesus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Review comment message cosine similarity

In [7]:
vectorizer = TfidfVectorizer(
    min_df=5,
    max_df=0.9,
    ngram_range=(1, 2),
    sublinear_tf=True,
    strip_accents="unicode",
    stop_words=list(stop_pt),
)

X = vectorizer.fit_transform(item_text["review_comment_message"])
print(X.shape)

from sklearn.neighbors import NearestNeighbors
import numpy as np

# X es tu matriz TF-IDF (sparse) con shape (n_items, n_terms)

# Entrenamos el índice de vecinos
nn = NearestNeighbors(metric="cosine", algorithm="brute")
nn.fit(X)

# Creamos un mapping item_id -> fila en X (para ubicar rápido el item)
item_ids = item_text["item_id"].values  # o la columna equivalente en tu DF final
id2idx = {item_id: i for i, item_id in enumerate(item_ids)}

print("Items indexados:", len(item_ids))

c:\Users\Jesus\anaconda3\envs\olist_project\lib\site-packages\sklearn\feature_extraction\text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ate', 'eramos', 'estao', 'estavamos', 'estiveramos', 'estivessemos', 'foramos', 'fossemos', 'ha', 'hao', 'houveramos', 'houverao', 'houveriamos', 'houvessemos', 'ja', 'sao', 'sera', 'serao', 'seriamos', 'so', 'tambem', 'tera', 'terao', 'teriamos', 'tinhamos', 'tiveramos', 'tivessemos', 'voce', 'voces'] not in stop_words.
  warnings.warn(


(11039, 7199)
Items indexados: 11039


In [8]:
id_to_idx = {iid: i for i, iid in enumerate(item_text["item_id"])}


def get_similar_items(item_id, top_k=10):
    """
    Devuelve una lista de vecinos similares para un item_id:
    [(neighbor_item_id, similarity_score), ...]
    """
    if item_id not in id2idx:
        return []

    idx = id2idx[item_id]

    # pedimos top_k+1 porque el primer vecino será el item mismo (distancia 0)
    distances, indices = nn.kneighbors(X[idx], n_neighbors=top_k + 1)

    distances = distances.ravel()
    indices = indices.ravel()

    results = []
    for dist, j in zip(distances, indices):
        if j == idx:
            continue  # salteamos el mismo item
        sim = 1.0 - dist  # cosine distance -> cosine similarity
        results.append((item_ids[j], float(sim)))

        if len(results) >= top_k:
            break

    return results

In [9]:
random_item = item_text.sample(1)["item_id"].values[0]
random_item

'8b395ea13c2e262687ff3e463c8867d4'

In [10]:
get_similar_items(random_item, top_k=5)

[('1136bf18450b05b755f50b49657c1943', 0.624800286438342),
 ('14bd7e575a12cf8543db3afd53a844bd', 0.6151262477644271),
 ('952e73f7e8bee61569a32b35c19bc981', 0.5570605274761409),
 ('f7ee078cfab4ad9144e1d75fd0258c5b', 0.521156678401819),
 ('e4f2856ddacde505a7d8a1faa4bf2c7d', 0.46884475687148686)]

In [11]:
products = pd.read_csv(PRODUCTS_PATH)[["product_id", "product_category_name"]]
tr = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")
cat = products.merge(tr, on="product_category_name", how="left")
cat = cat.rename(
    columns={"product_id": "item_id", "product_category_name_english": "category_en"}
)[["item_id", "category_en"]]

In [12]:
cat_map = dict(zip(cat["item_id"], cat["category_en"]))


def show_neighbors(item_id, neighbors):
    print("QUERY:", item_id, "| category:", cat_map.get(item_id))
    qtext = item_text.loc[
        item_text["item_id"] == item_id, "review_comment_message"
    ].iloc[0]
    print("Query text snippet:", qtext[:300], "\n")

    for nid, score in neighbors:
        print("  ->", nid, "| sim:", float(score), "| category:", cat_map.get(nid))
        ntext = item_text.loc[
            item_text["item_id"] == nid, "review_comment_message"
        ].iloc[0]
        print("     snippet:", ntext[:200])
        print()

In [13]:
neighbors = get_similar_items(random_item, top_k=5)
show_neighbors(random_item, neighbors)

QUERY: 8b395ea13c2e262687ff3e463c8867d4 | category: luggage_accessories
Query text snippet: adorei! estou feliz com o atendimento, o serviço prestado e o produto é ótimo. 

  -> 1136bf18450b05b755f50b49657c1943 | sim: 0.624800286438342 | category: electronics
     snippet: estou satisfeito e agradeço pelo bom serviço prestado. obrigado. elton.



  -> 14bd7e575a12cf8543db3afd53a844bd | sim: 0.6151262477644271 | category: sports_leisure
     snippet: estou muito satisfeito com o nível de serviço prestado. 

  -> 952e73f7e8bee61569a32b35c19bc981 | sim: 0.5570605274761409 | category: bed_bath_table
     snippet: bom atendimento prestado  muito bom não entrega o produto 

  -> f7ee078cfab4ad9144e1d75fd0258c5b | sim: 0.521156678401819 | category: perfumery
     snippet: serviço prestado dentro do esperado. obrigado tudo ok

  -> e4f2856ddacde505a7d8a1faa4bf2c7d | sim: 0.46884475687148686 | category: computers_accessories
     snippet: estou satisfeita com o serviço prestado. 
o meu pedido c